# Subteam A — Augmentation
**Dataset:** Fruit Detection — Roboflow (fruit-detection-2aajq/fruit-nxoac)

---
1. Translation
2. Rotation
3. Flipping
4. Cropping
5. Brightness and Contrast
6. Gamma Correction

- What it changes
- Why it helps
- Failure modes

## Step 0 — Import Libraries

In [ ]:
from __future__ import print_function
import cv2
import numpy as np
import matplotlib.pyplot as plt

## Step 1 — Load and Prepare Sample Image

In [ ]:
imagePath = "images/apple.jpg"
image = cv2.imread(imagePath)

# from bgr to rgb
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Get image dimensions
(h, w) = image.shape[:2]
print("Image shape (H x W x Channels):", image.shape)

# Resize to 640x640 with aspect ratio
scale = 640 / max(h, w)
newW = int(w * scale)
newH = int(h * scale)
resizedImage = cv2.resize(image, (newW, newH), interpolation=cv2.INTER_AREA)

# Place on canvas
canvas = np.full((640, 640, 3), 114, dtype=np.uint8)
padTop  = (640 - newH) // 2
padLeft = (640 - newW) // 2
canvas[padTop:padTop + newH, padLeft:padLeft + newW] = resizedImage

# Show the original image
plt.figure(figsize=(5, 5))
plt.imshow(canvas)
plt.title("Original Image (640x640)")
plt.axis("off")
plt.show()

print("Ready for augmentation!")

---
## Augmentation 1 — Translation

**What it changes:** Moves the image pixels in the X and Y directions by a fixed number of pixels.

**Why it helps:** fruits are rarely perfectly centred in the frame. Translation teaches the model to detect fruits at any place in the image

**Failure modes:** If the translation is too large, the fruit moves completely off-screen and the bounding box label becomes invalid.

In [ ]:
#Translation
# Translate 50 pixels right and 30 pixels down
translationMatrix1 = np.float32([[1, 0, 50], [0, 1, 30]])
translatedRight = cv2.warpAffine(canvas, translationMatrix1, (canvas.shape[1], canvas.shape[0]))

# Translate 50 pixels left and 30 pixels up
translationMatrix2 = np.float32([[1, 0, -50], [0, 1, -30]])
translatedLeft = cv2.warpAffine(canvas, translationMatrix2, (canvas.shape[1], canvas.shape[0]))

# Display
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(canvas)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(translatedRight)
plt.title("Translated Right (+50px, +30px)")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(translatedLeft)
plt.title("Translated Left (-50px, -30px)")
plt.axis("off")

plt.suptitle("Augmentation 1 - Translation", fontsize=14)
plt.tight_layout()
plt.savefig("aug1_translation.png")
plt.show()
print("Saved: aug1_translation.png")

---
## Augmentation 2 — Rotation

**What it changes:** Rotates the image around its center by a given angle.

**Why it helps:** Fruits on trees or in bowls appear at many angles. Rotation show the model that a banana is still a banana whether it's tilted 45 or 90 degrees.

**Failure modes:** Large rotations can rotate a fruit out of the frame, and the bounding box.

In [ ]:
#Rotation

(h, w) = canvas.shape[:2]
center = (w // 2, h // 2)

# Rotate 45 degrees
rotationMatrix45 = cv2.getRotationMatrix2D(center, 45, 1.0)
rotated45 = cv2.warpAffine(canvas, rotationMatrix45, (w, h))

# Rotate 90 degrees
rotationMatrix90 = cv2.getRotationMatrix2D(center, 90, 1.0)
rotated90 = cv2.warpAffine(canvas, rotationMatrix90, (w, h))

# Rotate -30 degrees
rotationMatrixMinus30 = cv2.getRotationMatrix2D(center, -30, 1.0)
rotatedMinus30 = cv2.warpAffine(canvas, rotationMatrixMinus30, (w, h))

# Display
plt.figure(figsize=(20, 5))

plt.subplot(1, 4, 1)
plt.imshow(canvas)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 4, 2)
plt.imshow(rotated45)
plt.title("Rotated 45 degrees")
plt.axis("off")

plt.subplot(1, 4, 3)
plt.imshow(rotated90)
plt.title("Rotated 90 degrees")
plt.axis("off")

plt.subplot(1, 4, 4)
plt.imshow(rotatedMinus30)
plt.title("Rotated -30 degrees")
plt.axis("off")

plt.suptitle("Augmentation 2 - Rotation", fontsize=14)
plt.tight_layout()
plt.savefig("aug2_rotation.png")
plt.show()
print("Saved: aug2_rotation.png")

---
## Augmentation 3 — Flipping

**What it changes:** Mirrors the image horizontally, vertically, or both.

**Why it helps:** Fruits can appear from any angle. A flipped apple is still an apple.

**Failure modes:** Flipping is not suitable for tasks where orientation matters.

In [ ]:
#Flipping

# Flip horizontally (left-right)
flippedHorizontal = cv2.flip(canvas, 1)

# Flip vertically (up-down)
flippedVertical = cv2.flip(canvas, 0)

# Flip both horizontally and vertically
flippedBoth = cv2.flip(canvas, -1)

# Display
plt.figure(figsize=(20, 5))

plt.subplot(1, 4, 1)
plt.imshow(canvas)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 4, 2)
plt.imshow(flippedHorizontal)
plt.title("Flipped Horizontally\n(cv2.flip = 1)")
plt.axis("off")

plt.subplot(1, 4, 3)
plt.imshow(flippedVertical)
plt.title("Flipped Vertically\n(cv2.flip = 0)")
plt.axis("off")

plt.subplot(1, 4, 4)
plt.imshow(flippedBoth)
plt.title("Flipped Both\n(cv2.flip = -1)")
plt.axis("off")

plt.suptitle("Augmentation 3 - Flipping", fontsize=14)
plt.tight_layout()
plt.savefig("aug3_flipping.png")
plt.show()
print("Saved: aug3_flipping.png")

---
## Augmentation 4 — Cropping

**What it changes:** Cuts out a region of the image using array slicing, removing parts of the image from the edges.

**Why it helps:** fruits are often partially hidden behind other fruits or cut off at the edge of the frame. Cropping teaches the model to detect partial fruits.

**Failure modes:** If the crop removes the fruit entirely, the image becomes negative.

In [ ]:
# Cropping

# Crop top-left region
croppedTopLeft = canvas[0:400, 0:400]

# Crop center region
croppedCenter = canvas[120:520, 120:520]

# Crop bottom-right region
croppedBottomRight = canvas[240:640, 240:640]

# Display
plt.figure(figsize=(20, 5))

plt.subplot(1, 4, 1)
plt.imshow(canvas)
plt.title("Original (640x640)")
plt.axis("off")

plt.subplot(1, 4, 2)
plt.imshow(croppedTopLeft)
plt.title("Crop Top-Left\n[0:400, 0:400]")
plt.axis("off")

plt.subplot(1, 4, 3)
plt.imshow(croppedCenter)
plt.title("Crop Center\n[120:520, 120:520]")
plt.axis("off")

plt.subplot(1, 4, 4)
plt.imshow(croppedBottomRight)
plt.title("Crop Bottom-Right\n[240:640, 240:640]")
plt.axis("off")

plt.suptitle("Augmentation 4 - Cropping", fontsize=14)
plt.tight_layout()
plt.savefig("aug4_cropping.png")
plt.show()
print("Saved: aug4_cropping.png")

---
## Augmentation 5 — Brightness and Contrast

**What it changes:** alpha controls contrast. beta controls brightness.

**Why it helps:** images have many lighting conditions. Brightness and contrast augmentation teaches the model to handle all of them.

**Failure modes:** Setting brightness too high will wash out the image and lose all texture detail. Setting contrast too low makes the image too dark.

In [ ]:
#Brightness and Contrast

# Brighter image
brighterImage = cv2.convertScaleAbs(canvas, alpha=1.0, beta=60)

# Darker image
darkerImage = cv2.convertScaleAbs(canvas, alpha=1.0, beta=-60)

# Higher contrast
highContrastImage = cv2.convertScaleAbs(canvas, alpha=1.5, beta=0)

# Lower contrast
lowContrastImage = cv2.convertScaleAbs(canvas, alpha=0.5, beta=0)

# Display
plt.figure(figsize=(25, 5))

plt.subplot(1, 5, 1)
plt.imshow(canvas)
plt.title("Original\nalpha=1.0, beta=0")
plt.axis("off")

plt.subplot(1, 5, 2)
plt.imshow(brighterImage)
plt.title("Brighter\nalpha=1.0, beta=60")
plt.axis("off")

plt.subplot(1, 5, 3)
plt.imshow(darkerImage)
plt.title("Darker\nalpha=1.0, beta=-60")
plt.axis("off")

plt.subplot(1, 5, 4)
plt.imshow(highContrastImage)
plt.title("High Contrast\nalpha=1.5, beta=0")
plt.axis("off")

plt.subplot(1, 5, 5)
plt.imshow(lowContrastImage)
plt.title("Low Contrast\nalpha=0.5, beta=0")
plt.axis("off")

plt.suptitle("Augmentation 5 - Brightness and Contrast", fontsize=14)
plt.tight_layout()
plt.savefig("aug5_brightness_contrast.png")
plt.show()
print("Saved: aug5_brightness_contrast.png")

---
## Augmentation 6 — Gamma Correction

**What it changes:** Gamma correction applies a non-linear transformation to pixel intensities. Gamma < 1 brightens the image. Gamma > 1 darkens the image.

**Why it helps:** Unlike brightness, gamma affects dark and bright regions differently similar to how cameras and phone sensors capture images differently.

**Failure modes:** Very low gamma values overexpose the image and wash out all fruit colors. Very high gamma values make the image nearly black, destroying all visible features.

In [ ]:
#Gamma Correction

# Gamma 0.4
gamma04 = 0.4
lookupTable04 = np.array([((i / 255.0) ** (1.0 / gamma04)) * 255
                          for i in np.arange(0, 256)]).astype("uint8")
gammaImage04 = cv2.LUT(canvas, lookupTable04)

# Gamma 1.0
gamma10 = 1.0
lookupTable10 = np.array([((i / 255.0) ** (1.0 / gamma10)) * 255
                          for i in np.arange(0, 256)]).astype("uint8")
gammaImage10 = cv2.LUT(canvas, lookupTable10)

# Gamma 1.8
gamma18 = 1.8
lookupTable18 = np.array([((i / 255.0) ** (1.0 / gamma18)) * 255
                          for i in np.arange(0, 256)]).astype("uint8")
gammaImage18 = cv2.LUT(canvas, lookupTable18)

# Gamma 2.5 
gamma25 = 2.5
lookupTable25 = np.array([((i / 255.0) ** (1.0 / gamma25)) * 255
                          for i in np.arange(0, 256)]).astype("uint8")
gammaImage25 = cv2.LUT(canvas, lookupTable25)

# Display
plt.figure(figsize=(20, 5))

plt.subplot(1, 4, 1)
plt.imshow(gammaImage04)
plt.title("Gamma = 0.4\n(Very Bright)")
plt.axis("off")

plt.subplot(1, 4, 2)
plt.imshow(gammaImage10)
plt.title("Gamma = 1.0\n(Original)")
plt.axis("off")

plt.subplot(1, 4, 3)
plt.imshow(gammaImage18)
plt.title("Gamma = 1.8\n(Slightly Dark)")
plt.axis("off")

plt.subplot(1, 4, 4)
plt.imshow(gammaImage25)
plt.title("Gamma = 2.5\n(Dark)")
plt.axis("off")

plt.suptitle("Augmentation 6 - Gamma Correction", fontsize=14)
plt.tight_layout()
plt.savefig("aug6_gamma.png")
plt.show()
print("Saved: aug6_gamma.png")

---
## All 6 Augmentations — Side by Side Summary

In [ ]:
# Display all 6 augmentations together in one figure
plt.figure(figsize=(25, 10))

plt.subplot(2, 4, 1)
plt.imshow(canvas)
plt.title("Original")
plt.axis("off")

plt.subplot(2, 4, 2)
plt.imshow(translatedRight)
plt.title("1. Translation\n(+50px right, +30px down)")
plt.axis("off")

plt.subplot(2, 4, 3)
plt.imshow(rotated45)
plt.title("2. Rotation\n(45 degrees)")
plt.axis("off")

plt.subplot(2, 4, 4)
plt.imshow(flippedHorizontal)
plt.title("3. Flipping\n(Horizontal)")
plt.axis("off")

plt.subplot(2, 4, 5)
plt.imshow(croppedCenter)
plt.title("4. Cropping\n(Center region)")
plt.axis("off")

plt.subplot(2, 4, 6)
plt.imshow(brighterImage)
plt.title("5. Brightness\n(beta=+60)")
plt.axis("off")

plt.subplot(2, 4, 7)
plt.imshow(highContrastImage)
plt.title("5. Contrast\n(alpha=1.5)")
plt.axis("off")

plt.subplot(2, 4, 8)
plt.imshow(gammaImage04)
plt.title("6. Gamma\n(gamma=0.4)")
plt.axis("off")

plt.suptitle("All 6 Augmentations — Summary", fontsize=16)
plt.tight_layout()
plt.savefig("aug_all_summary.png")
plt.show()
print("Saved: aug_all_summary.png")